In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src import preprocessing, features

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')
FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future



In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = preprocessing.process_store_data(store_df)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
sales_df = train_df[['Date', 'Store', 'Sales']].copy()
train_df.drop(['Customers', 'Sales'], axis=1)

targets = features.make_targets(sales_df, horizon=FORECAST_HORIZON)

test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_3132\1651772828.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])


In [3]:
""" Feature engineering """
lags = [1, 2, 3, 4, 5, 6, 7] # days


train_df = features.attach_store_data(train_df, store_df)

# Competition-related features
train_df['CompetitionDistance'] = train_df['CompetitionDistance'].apply(np.log1p)
train_df['CompetitionSinceMonths'] = ( (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days / 30.0 ).round()

# Promotion related features
train_df['Promo2SinceWeeks'] =  ( (train_df['Date'] - train_df['Promo2SinceDate']).dt.days / 7.0 ).fillna(0).round().astype(int) * train_df['Promo2']

# Basic date features
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week
train_df['Month'] = train_df['Date'].dt.month
train_df['Year'] = train_df['Date'].dt.year
train_df['Quarter'] = train_df['Date'].dt.quarter

# Calendar and seasonality features
train_df['is_weekend'] = train_df['Date'].dt.dayofweek >= 5

# Cyclical features
train_df['Month_sin'] = np.sin(2 * np.pi * train_df['Month'] / 12)
train_df['Month_cos'] = np.cos(2 * np.pi * train_df['Month'] / 12)
train_df['Dayofweek_sin'] = np.sin(2 * np.pi * train_df['DayOfWeek'] / 7)
train_df['Dayofweek_cos'] = np.cos(2 * np.pi * train_df['DayOfWeek'] / 7)

# Lagged features
date_lags = [DateOffset(days=lag) for lag in lags]
lagged_df = features.make_lags(sales_df, date_lags)

# Rolling-window features

# Drop useless
#train_df.drop(['Promo2SinceDate', 'CompetitionSinceDate'], axis=1, inplace=True)


In [4]:
train_df.merge(lagged_df, how='left').head(3)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Month_cos,Dayofweek_sin,Dayofweek_cos,lag_days_1,lag_days_2,lag_days_3,lag_days_4,lag_days_5,lag_days_6,lag_days_7
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,-0.866025,-0.974928,-0.222521,5020.0,4782.0,5011.0,6102.0,0.0,4364.0,3706.0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,-0.866025,-0.974928,-0.222521,5567.0,6402.0,5671.0,6627.0,0.0,2512.0,3854.0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,-0.866025,-0.974928,-0.222521,8977.0,7610.0,8864.0,8107.0,0.0,3878.0,5080.0


In [5]:

# TODO: Lag and rolling features - NOTE: We are predicting 6 weeks ahead.
""" 
train_df['lag_1'] = train_df['target'].shift(1)
train_df['rolling_mean_7'] = train_df['target'].shift(1).rolling(7).mean()
train_df['rolling_std_7'] = train_df['target'].shift(1).rolling(7).std()
"""

" \ntrain_df['lag_1'] = train_df['target'].shift(1)\ntrain_df['rolling_mean_7'] = train_df['target'].shift(1).rolling(7).mean()\ntrain_df['rolling_std_7'] = train_df['target'].shift(1).rolling(7).std()\n"

In [18]:
df_pivot = (pd.pivot(sales_df,
                     index=sales_df.columns[0],
                     columns=sales_df.columns[1],
                     values=sales_df.columns[2])
                .sort_index() # Sorted from oldest to newest
                .shift(freq=DateOffset(days=1))
                #.rolling(window=DateOffset(weeks=1))
                #.mean()
                .reset_index()
                       )

df_pivot.head(5)


Store,Date,1,2,3,4,5,6,7,8,9,...,1106,1107,1108,1109,1110,1111,1112,1113,1114,1115
0,2013-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2013-01-03,5530.0,4422.0,6823.0,9941.0,4253.0,6089.0,8244.0,5419.0,4903.0,...,5099.0,3955.0,6220.0,4576.0,4126.0,5097.0,10797.0,6218.0,20642.0,3697.0
2,2013-01-04,4327.0,4159.0,5902.0,8247.0,3465.0,5398.0,7231.0,4842.0,4602.0,...,4330.0,3151.0,4779.0,3654.0,3508.0,4579.0,8716.0,5563.0,18463.0,4297.0
3,2013-01-05,4486.0,4484.0,6069.0,8290.0,4456.0,6092.0,7758.0,4059.0,4798.0,...,3956.0,3990.0,5491.0,3596.0,3933.0,4640.0,9788.0,5524.0,18371.0,4540.0
4,2013-01-06,4997.0,2342.0,4523.0,10338.0,1590.0,3872.0,5218.0,2337.0,4254.0,...,2624.0,5128.0,2113.0,2897.0,3156.0,3325.0,9513.0,5194.0,18856.0,4771.0
